Import Python modules

In [1]:
import os
import bte
from collections import defaultdict
from collections import Counter
import pandas as pd
import numpy as np
import math

import sys
sys.path.append('../scripts/')
import MATWrapper
import ExpectedCalc

resultsdir = '../results/'
if not os.path.isdir(resultsdir):
    os.makedirs(resultsdir)

**Here are steps that I did manually that aren't yet part of the notebook.**

First, I downloaded the tree from an UShER website for influenza trees: https://hgwdev-angie.gi.ucsc.edu/cgi-bin/hgPhyloPlace?hgpp_org=influenzaA

The notebook analyzes data for the HA segment from H3N2, with the root A/New York/392/2004(H3N2). To get to the data, just click the link in the column called "Download directory", then download the `.pb` and metadata files.

Second, I re-rooted the tree. The current root is a sequence from 2004. I rerooted it to a sequence from 1968.

The script that I used to do this is `data/reroot.sh`.

Here's a tricky thing. We need to reroot the tree for each genome segment. We'd probably like to use the same root for all of them. A goal is to find: what is a 1968ish sequence that is common to all segments?

**Another thing to consider with the input tree is ways to filter it**
* remove sequences that...
    * occur before 1968
    * are shorter than some length cutoff. I don't know what exactly we should use, but the HA coding sequnce is 1,700 nucleotides and the full segment is longer than that by a bit
    * prune away non-human sequences (see metadata file)

Read in the original and rerooted trees

In [2]:
otree = bte.MATree('../data/fluA.GCF_000865085.1.NC_007366.1.latest.pb.gz')
rtree = bte.MATree('../data/rerooted_fluA.GCF_000865085.1.NC_007366.1.latest.pb')

Finished 'from_pb' in 1.2287 seconds
Finished 'from_pb' in 1.1613 seconds


Make sure that the two trees have different roots

In [3]:
otree.root

id: node_1
level: 1
parent: None
children: ['node_2', 'A/New_York/392/2004_H3N2|CY002064.1|2004-12-21', 'A/New_York/392/2004_H3N2|NC_007366.1|2004-12-21']
mutations: []
annotations: []
branch length: 0.0

In [4]:
rtree.root

id: node_1
level: 1
parent: None
children: ['node_2']
mutations: []
annotations: []
branch length: 0.0

Check the mutations associated with a given node, doing so separately for each tree.

In [5]:
node_id = 'A/mallard/Wisconsin/260/1980_H3N2|CY178390.1|1980' # 'A/Bcm/1/1970_H3N2|CY111497.1|1970-06-14' # 'A/Aichi/2/68_H3N2/1968|EF614251.1|1968' #
otree.get_node(node_id)

id: A/mallard/Wisconsin/260/1980_H3N2|CY178390.1|1980
level: 125
parent: node_21953
children: []
mutations: ['C220A', 'T335C', 'C350T', 'C389G', 'G507A', 'C572T', 'G1044A', 'C1193T', 'T1538C']
annotations: []
branch length: 9.0

In [6]:
rtree.get_node(node_id)

id: A/mallard/Wisconsin/260/1980_H3N2|CY178390.1|1980
level: 21
parent: node_1270
children: []
mutations: ['C220A', 'T335C', 'C350T', 'C389G', 'G507A', 'C572T', 'G1044A', 'C1193T', 'T1538C']
annotations: []
branch length: 9.0

To see if the rerooting command flipped the mutations that should have been flipped, compute paths between a tree's root and the other tree's root sequence. I think that all mutations along this path should be flipped (and only mutations along this path should be flipped).

In [7]:
new_root = 'A/Aichi/2/68_H3N2/1968|EF614251.1|1968'
path_to_new_root = set(otree.get_haplotype(new_root))
print(len(path_to_new_root))

208


In [8]:
old_root = 'A/New_York/392/2004_H3N2|CY002064.1|2004-12-21' # this sequence's parent is node_1 (no mutations relative to parent)
path_to_old_root = set(rtree.get_haplotype(old_root))
print(len(path_to_old_root))

208


There should be no mutations in common between the above two sets.

In [9]:
set.intersection(path_to_new_root, path_to_old_root)

set()

The mutations in the first set should all be flipped versions of the mutations in the second set.

In [10]:
rev_path_to_new_root = set([x[-1] + x[1:-1] + x[0] for x in path_to_new_root])
len(set.intersection(rev_path_to_new_root, path_to_old_root))

197

Most mutations meet the above expectations, but a few do not, shown below. I don't know why this is.

In [11]:
sorted(list(rev_path_to_new_root.difference(path_to_old_root)))

['A553T',
 'A753G',
 'A862G',
 'C428A',
 'C439A',
 'C821A',
 'G1343A',
 'G910A',
 'T1364G',
 'T185C',
 'T956A']

In [12]:
[mut for mut in rev_path_to_new_root if '553' in mut]

['A553T']

In [13]:
[mut for mut in path_to_new_root if '553' in mut]

['T553A']

In [14]:
[mut for mut in path_to_old_root if '553' in mut]

['G553A']

Get the old reference nucleotide sequence from NCBI

In [15]:
df = pd.read_csv('../data/fluA.GCF_000865085.1.NC_007366.1.latest.metadata.tsv.gz', sep='\t')
df[df['strain'] == old_root]

,strain,genbank_accession,date,country,location,length,host,bioproject_accession,biosample_accession,sra_accession,authors,publications,serotype,segment,Nextstrain_clade
39635,A/New_York/392/2004_H3N2|CY002064.1|2004-12-21,CY002064.1,2004-12-21,USA,"Tompkins County, NY",1762,Homo sapiens,NaN,NaN,NaN,"Ghedin,E., Miller,N., Zaborsky,J., Feldblyum,T...",NaN,H3N2,HA,unassigned


In [16]:
def fasta_to_seq(fasta_file):
    with open(fasta_file) as f:
        lines = f.readlines()
        seq = ''
        for line in lines[1:]:
            seq += line.strip()
    return seq
old_ref_seq = list(fasta_to_seq('../data/CY002064.1.fasta'))
len(old_ref_seq)

1762

Infer the sequence of the new root and write this sequence to a FASTA file

In [17]:
inferred_new_ref_seq = old_ref_seq.copy()
for mut in path_to_new_root:
    (wt_nt, site, mut_nt) = (mut[0], int(mut[1:-1]), mut[-1])
    #print(mut, wt_nt, site, mut_nt)
    assert old_ref_seq[site-1] == wt_nt, site
    assert inferred_new_ref_seq[site-1] == wt_nt, site
    inferred_new_ref_seq[site-1] = mut_nt

output_fasta = os.path.join(resultsdir, 'inferred_new_ref_seq.fasta')
if not os.path.isfile(output_fasta):
    with open(output_fasta, 'w') as f:
        f.write('>inferred_new_ref_seq\n')
        f.write(''.join(inferred_new_ref_seq))

Does the inferred reference sequence match the expected one?

In [18]:
df[df['strain'] == new_root]

,strain,genbank_accession,date,country,location,length,host,bioproject_accession,biosample_accession,sra_accession,authors,publications,serotype,segment,Nextstrain_clade
102,A/Aichi/2/68_H3N2/1968|EF614251.1|1968,EF614251.1,1968,Singapore,NaN,1698,Homo sapiens,NaN,NaN,NaN,"Narasaraju,T., Sim,M.K., Ng,H.H., Phoon,M.C., ...",18983930,H3N2,HA,unassigned


In [19]:
new_ref_seq = list(fasta_to_seq('../data/EF614251.1.fasta'))
print(len(new_ref_seq))
for (site, (i_nt, a_nt)) in enumerate(zip(new_ref_seq, inferred_new_ref_seq[30-1:1730-1])):
    if i_nt != a_nt:
        print(site, i_nt, a_nt)

1698
155 G T
398 T C
409 T C
523 G A
723 C A
791 T C
832 C A
880 T G
926 C T
1313 T G
1334 C T


## Hey Chris, here's an important part of the code that makes a dataframe with all possible single-nucleotide mutations to the reference sequence

This is kind of hacky because it uses the inferred sequence of the new root, which is not totally correct. But we can correct it in the future.

Make a file with metadata on each site

In [ ]:
ref_seq = ''.join(inferred_new_ref_seq) # here's the close but not correct reference sequence
coding_sites_dict = defaultdict(list)
gene = 'HA'
gene_start = 30
gene_end = 1730
for site in range(1, len(ref_seq)+1):
    if (site < gene_start) or (site > gene_end):
        coding_sites_dict['site'].append(site)
        coding_sites_dict['codon_position'].append('noncoding')
        coding_sites_dict['codon_site'].append('noncoding')
        coding_sites_dict['gene'].append('noncoding')
    else:
        coding_sites_dict['site'].append(site)
        remainder = (site-gene_start+1)%3
        if remainder == 0:
            coding_sites_dict['codon_position'].append(3)
        else:
            coding_sites_dict['codon_position'].append(remainder)
        coding_sites_dict['codon_site'].append(math.ceil((site-gene_start+1)/3))
        coding_sites_dict['gene'].append(gene)

coding_sites_df = pd.DataFrame(coding_sites_dict)

coding_sites_file = os.path.join(resultsdir, f'coding_sites_{gene}.csv')
if not os.path.isfile(coding_sites_file):
    coding_sites_df.to_csv(coding_sites_file, index=False)

Make a dataframe with all possible single nucleotide mutations to the reference sequence

In [21]:
ref_fasta = os.path.join(resultsdir, 'inferred_new_ref_seq.fasta')
possible_mutations_object = ExpectedCalc.PossibleMutations(ref_fasta, coding_sites_file)
ref_seq = possible_mutations_object.ref_seq
ref_df = possible_mutations_object.possible_mutations_df()
ref_df.rename(columns={'codon_sites':'codon_site'}, inplace=True)
ref_df['gene_codon_site_id'] = ref_df['gene'] + '-' + ref_df['codon_site'].astype(str)
def get_3mer_seq_context(nt_site, seq):
    return seq[nt_site-2:nt_site+1]
ref_df['seq_context'] = ref_df['site'].apply(lambda x: get_3mer_seq_context(int(x), ref_seq))
del ref_df['ID']
ref_df.head(n=12)

,site,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,mut_aa,is_synonymous,nuc,aa,gene_codon_site_id,seq_context
87,30,A,G,HA,1,1,ATG,GTG,M,V,False,A30G,M1V,HA-1,CAT
88,30,A,C,HA,1,1,ATG,CTG,M,L,False,A30C,M1L,HA-1,CAT
89,30,A,T,HA,1,1,ATG,TTG,M,L,False,A30T,M1L,HA-1,CAT
90,31,T,G,HA,2,1,ATG,AGG,M,R,False,T31G,M1R,HA-1,ATG
91,31,T,C,HA,2,1,ATG,ACG,M,T,False,T31C,M1T,HA-1,ATG
92,31,T,A,HA,2,1,ATG,AAG,M,K,False,T31A,M1K,HA-1,ATG
93,32,G,T,HA,3,1,ATG,ATT,M,I,False,G32T,M1I,HA-1,TGA
94,32,G,C,HA,3,1,ATG,ATC,M,I,False,G32C,M1I,HA-1,TGA
95,32,G,A,HA,3,1,ATG,ATA,M,I,False,G32A,M1I,HA-1,TGA
96,33,A,G,HA,1,2,AAG,GAG,K,E,False,A33G,K2E,HA-2,GAA


## Count mutations along the branches of the rerooted tree.

In [22]:
tree = rtree
ref_fasta = os.path.join(resultsdir, 'inferred_new_ref_seq.fasta')
ref_gtf = '../data/prelim_ref_ha.gtf'
translations = tree.translate(ref_gtf, ref_fasta)

In [23]:
# Cycle over nodes and record codon mutation counts
nodes = tree.depth_first_expansion()
print('Number of nodes in tree:', len(nodes))
nt_mut_counts = defaultdict(lambda: 0)
codon_mut_counts = defaultdict(lambda: 0)
conservation_nt_mut_counter = Counter()
conservation_codon_mut_counter = Counter()
n_nodes_passing_filters = 0
n_internal_nodes_analyzed = 0
for (i, node) in enumerate(nodes):
    
    if node.parent == None:
        continue

    # Get lists of nt-level and codon-level mutations
    nt_muts = node.mutations
    if node.id not in translations:
        codon_muts = []
    else:
        codon_muts = translations[node.id]

    # Ignore nodes with more than four mutations
    if len(nt_muts) > 4:
        continue

    # Ignore nodes with more than one reversion to the reference
    reversions_to_reference = [
        nt_mut for nt_mut in nt_muts
        if ref_seq[int(nt_mut[1:-1])-1] == nt_mut[-1]
    ]
    if len(reversions_to_reference) > 1:
        continue

    # Ignore nodes with more than one mutation to the same codon
    if len(codon_muts) > 1:
        if max(Counter((codon_mut.gene, codon_mut.aa_index) for codon_mut in codon_muts).values()) > 1:
            continue
    
    # The node has passed the above filters
    n_nodes_passing_filters += 1
    
    # Reconstruct the parent node's genome for use in getting each mutation's
    # sequence context
    node_haplotype = tree.get_haplotype(node.id)
    parent_node_haplotype = tree.get_haplotype(node.parent.id)
    parent_node_seq = MATWrapper.apply_muts(ref_seq, parent_node_haplotype)

    # Record counts of nt-level mutations
    for nt_mut in nt_muts:
        seq_context = get_3mer_seq_context(int(nt_mut[1:-1]), parent_node_seq)
        ref_seq_context = get_3mer_seq_context(int(nt_mut[1:-1]), ref_seq)
        nt_mut_id = f'{nt_mut}-{seq_context}-{ref_seq_context}'
        nt_mut_counts[nt_mut_id] += 1

    # Record counts of codon-level mutations
    for codon_mut in codon_muts:
        nt_mut = codon_mut.nuc
        seq_context = get_3mer_seq_context(int(nt_mut[1:-1]), parent_node_seq)
        ref_seq_context = get_3mer_seq_context(int(nt_mut[1:-1]), ref_seq)
        codon_mut_id = f'{codon_mut.gene}-{codon_mut.aa_index}-{codon_mut.original_codon}-{codon_mut.alternative_codon}-{seq_context}-{ref_seq_context}'
        codon_mut_counts[codon_mut_id] += 1

    # Record rolling counts of all nt-level and codon-level mutations in a node
    # relative to the reference sequence in order to determine the level of
    # conservation at each site. Only do this for internal nodes.
    if not node.is_leaf():
        n_internal_nodes_analyzed += 1
        conservation_nt_mut_counter += Counter([
            mut[1:-1] for mut in node_haplotype
        ])
        conservation_codon_mut_counter += Counter(set(ref_df[
            ref_df['nuc'].isin(node_haplotype)
        ]['gene_codon_site_id']))

    if i % 10000 == 0:
        print(i)
    # if len(nt_muts) == 0:
    #     continue
#     if i > 100:
#         break
    
print('Number of nodes that passed filters:', n_nodes_passing_filters)
print('Number of internal nodes analyzed:', n_internal_nodes_analyzed)

# Store mutation counts in dataframe
nt_mut_counts_dict = defaultdict(list)
for (nt_mut, counts) in nt_mut_counts.items():
    nt_mut_counts_dict['nt_mut_id'].append(nt_mut)
    nt_mut_counts_dict['counts'].append(counts)
nt_mut_counts_df = pd.DataFrame(nt_mut_counts_dict)
nt_mut_counts_df[[
    'nt_mut', 'mut_seq_context', 'ref_seq_context'
]] = nt_mut_counts_df['nt_mut_id'].str.extract('(\w+)-(\w+)-(\w+)')
del nt_mut_counts_df['nt_mut_id']

codon_mut_counts_dict = defaultdict(list)
for (codon_mut, counts) in codon_mut_counts.items():
    codon_mut_counts_dict['codon_mut_id'].append(codon_mut)
    codon_mut_counts_dict['counts'].append(counts)
codon_mut_counts_df = pd.DataFrame(codon_mut_counts_dict)
codon_mut_counts_df[[
    'gene', 'codon_site', 'wt_codon', 'mut_codon', 'mut_seq_context', 'ref_seq_context'
]] = codon_mut_counts_df['codon_mut_id'].str.extract('(\w+)-(\d+)-(\w+)-(\w+)-(\w+)-(\w+)')
del codon_mut_counts_df['codon_mut_id']

# Write mutation counts to output file
nt_mut_counts_df.to_csv(os.path.join(resultsdir, 'nt_mut_id_counts.csv'), index=False)
codon_mut_counts_df.to_csv(os.path.join(resultsdir, 'codon_mut_id_counts.csv'), index=False)

# Output data in conservation counters as dataframes
counters = [
    ('nt', conservation_nt_mut_counter),
    ('codon', conservation_codon_mut_counter),
]
for (counter_id, counter) in counters:
    df = pd.DataFrame.from_dict(
        counter, orient='index'
    ).reset_index().rename(columns={
        'index' : 'site',
        0 : 'count'
    })
    df['frac_mut'] = df['count'] / n_internal_nodes_analyzed
    df['frac_cons'] = 1 - df['frac_mut']
    df.sort_values('count', ascending=False, inplace=True)
    df.to_csv(os.path.join(resultsdir, f'{counter_id}_conservation.csv'), index=False)

codon_mut_counts_df.head()

Number of nodes in tree: 103143


/fh/fast/matsen_e/hhaddox/2025/flu-syn-rates/notebooks/../scripts/MATWrapper.py:96: UserWarning: parent base doesn't match existing base at the sequence
  warn("parent base doesn't match existing base at the sequence")


10000
20000
30000
50000
60000
70000
80000
90000
100000
Number of nodes that passed filters: 96196
Number of internal nodes analyzed: 24232


,counts,gene,codon_site,wt_codon,mut_codon,mut_seq_context,ref_seq_context
0,14,HA,161,AGC,AAC,AGC,AGC
1,1,HA,78,ATA,AGA,ATA,ATA
2,18,HA,163,TTT,TTC,TTT,TTT
3,1,HA,160,GCC,ACC,TGC,TGG
4,38,HA,482,TGT,TGC,GTT,GCT


Merge the dataframe of all possible mutations with the dataframe of counts

In [24]:
counts_df = (
    ref_df
    .merge(codon_mut_counts_df, on=['codon_site', 'wt_codon', 'mut_codon', 'gene'], how='left')
)
counts_df['mut_type'] = counts_df['wt_nt'] + counts_df['mut_nt']
counts_df['codon_site'] = counts_df['codon_site'].astype(int)
counts_df['counts'] = counts_df['counts'].fillna(0)
counts_df['counts'] = counts_df['counts'].astype(int)
counts_df.head()

,site,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,mut_aa,is_synonymous,nuc,aa,gene_codon_site_id,seq_context,counts,mut_seq_context,ref_seq_context,mut_type
0,30,A,G,HA,1,1,ATG,GTG,M,V,False,A30G,M1V,HA-1,CAT,0,NaN,NaN,AG
1,30,A,C,HA,1,1,ATG,CTG,M,L,False,A30C,M1L,HA-1,CAT,0,NaN,NaN,AC
2,30,A,T,HA,1,1,ATG,TTG,M,L,False,A30T,M1L,HA-1,CAT,0,NaN,NaN,AT
3,31,T,G,HA,2,1,ATG,AGG,M,R,False,T31G,M1R,HA-1,ATG,0,NaN,NaN,TG
4,31,T,C,HA,2,1,ATG,ACG,M,T,False,T31C,M1T,HA-1,ATG,0,NaN,NaN,TC


In [25]:
counts_df[counts_df['codon_site'] == 3]

,site,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,mut_aa,is_synonymous,nuc,aa,gene_codon_site_id,seq_context,counts,mut_seq_context,ref_seq_context,mut_type
20,36,A,G,HA,1,3,ACC,GCC,T,A,False,A36G,T3A,HA-3,GAC,2,GAC,GAC,AG
21,36,A,C,HA,1,3,ACC,CCC,T,P,False,A36C,T3P,HA-3,GAC,0,NaN,NaN,AC
22,36,A,T,HA,1,3,ACC,TCC,T,S,False,A36T,T3S,HA-3,GAC,1,GAC,GAC,AT
23,37,C,T,HA,2,3,ACC,ATC,T,I,False,C37T,T3I,HA-3,ACC,4,ACC,ACC,CT
24,37,C,G,HA,2,3,ACC,AGC,T,S,False,C37G,T3S,HA-3,ACC,0,NaN,NaN,CG
25,37,C,A,HA,2,3,ACC,AAC,T,N,False,C37A,T3N,HA-3,ACC,1,ACC,ACC,CA
26,38,C,T,HA,3,3,ACC,ACT,T,T,True,C38T,T3T,HA-3,CCA,4,CCG,CCA,CT
27,38,C,T,HA,3,3,ACC,ACT,T,T,True,C38T,T3T,HA-3,CCA,6,CCA,CCA,CT
28,38,C,G,HA,3,3,ACC,ACG,T,T,True,C38G,T3T,HA-3,CCA,0,NaN,NaN,CG
29,38,C,A,HA,3,3,ACC,ACA,T,T,True,C38A,T3T,HA-3,CCA,1,CCA,CCA,CA


Get data for synonymous mutations

In [26]:
syn_counts_df = counts_df[counts_df['is_synonymous'] == True]
syn_counts_df.to_csv(os.path.join(resultsdir, 'syn_counts.csv'), index=False)
syn_counts_df.head()

,site,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,mut_aa,is_synonymous,nuc,aa,gene_codon_site_id,seq_context,counts,mut_seq_context,ref_seq_context,mut_type
18,35,G,A,HA,3,2,AAG,AAA,K,K,True,G35A,K2K,HA-2,AGA,43,AGA,AGA,GA
19,35,G,A,HA,3,2,AAG,AAA,K,K,True,G35A,K2K,HA-2,AGA,9,AGG,AGA,GA
26,38,C,T,HA,3,3,ACC,ACT,T,T,True,C38T,T3T,HA-3,CCA,4,CCG,CCA,CT
27,38,C,T,HA,3,3,ACC,ACT,T,T,True,C38T,T3T,HA-3,CCA,6,CCA,CCA,CT
28,38,C,G,HA,3,3,ACC,ACG,T,T,True,C38G,T3T,HA-3,CCA,0,NaN,NaN,CG


## Chris, for the analysis on how conserved each site is, I remember seeing that about half of sites had not very good conservation.

Going forward, we might have to consider something where we chop up the tree into pieces and sites are conserved within each piece. And then when we compare counts across mutations, we need to account for evolutionary opportunity.